In [ ]:
# ============== SETUP: run this cell first ==============
import os, sys, time, json, gc, subprocess
from pathlib import Path

import torch
assert torch.cuda.is_available(), "Need a GPU. Runtime > Change runtime type > A100 GPU."
GPU_NAME = torch.cuda.get_device_name(0)
print(f"GPU: {GPU_NAME} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")

# Install latest transformers + bnb
print("installing dependencies...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                       "transformers", "accelerate", "bitsandbytes", "matplotlib"])
import transformers
print(f"transformers: {transformers.__version__}")

import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ===== Helper: find all RMSNorm modules in a model =====
def find_rmsnorm_modules(model):
    """Return list of (name, module) for all RMSNorm-like modules."""
    norms = []
    for name, module in model.named_modules():
        type_name = type(module).__name__
        if 'RMSNorm' in type_name:
            norms.append((name, module))
    return norms

# ===== Helper: per-module CUDA-event timing via hooks =====
class NormProfiler:
    def __init__(self):
        self.events = []   # list of (start_event, end_event)
        self.handles = []
    def attach(self, modules):
        for _, mod in modules:
            handles = [
                mod.register_forward_pre_hook(self._pre),
                mod.register_forward_hook(self._post),
            ]
            self.handles.extend(handles)
    def _pre(self, module, input):
        ev = torch.cuda.Event(enable_timing=True); ev.record()
        module._prof_start = ev
    def _post(self, module, input, output):
        ev = torch.cuda.Event(enable_timing=True); ev.record()
        self.events.append((module._prof_start, ev))
    def detach(self):
        for h in self.handles:
            h.remove()
        self.handles = []
    def total_ms(self):
        torch.cuda.synchronize()
        return sum(s.elapsed_time(e) for s, e in self.events)
    def reset(self):
        self.events = []

def free_model(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print(f"  GPU memory after free: {torch.cuda.memory_allocated()/1e9:.2f} GB")

def load_at_precision(model_id, precision):
    """Return (model, tokenizer) loaded at the given precision."""
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if precision == "bf16":
        model = AutoModelForCausalLM.from_pretrained(
            model_id, torch_dtype=torch.bfloat16, device_map="cuda",
            attn_implementation="eager", trust_remote_code=True
        ).eval()
    elif precision == "8bit":
        bnb = BitsAndBytesConfig(load_in_8bit=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_id, quantization_config=bnb, device_map="cuda",
            attn_implementation="eager", trust_remote_code=True
        ).eval()
    elif precision == "4bit":
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_compute_dtype=torch.bfloat16,
                                 bnb_4bit_use_double_quant=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_id, quantization_config=bnb, device_map="cuda",
            attn_implementation="eager", trust_remote_code=True
        ).eval()
    else:
        raise ValueError(f"unknown precision: {precision}")
    return model, tokenizer

def profile_precision(model_id, precision, prompt, max_tokens, n_warmup, n_trials):
    """Load model at precision, profile RMSNorm time vs total time, return summary."""
    print(f"\n========== {precision} on {model_id} ==========")
    t0 = time.time()
    model, tokenizer = load_at_precision(model_id, precision)
    print(f"  loaded in {time.time()-t0:.1f}s, GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")
    norms = find_rmsnorm_modules(model)
    print(f"  found {len(norms)} RMSNorm modules")
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    # Warmup
    for _ in range(n_warmup):
        with torch.no_grad():
            _ = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False, use_cache=True)
        torch.cuda.synchronize()
    # Profile each trial
    prof = NormProfiler()
    prof.attach(norms)
    total_ms_list = []
    norm_ms_list  = []
    for trial in range(n_trials):
        prof.reset()
        torch.cuda.synchronize()
        t0 = time.time()
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False, use_cache=True)
        torch.cuda.synchronize()
        total_ms = (time.time() - t0) * 1000
        norm_ms = prof.total_ms()
        total_ms_list.append(total_ms)
        norm_ms_list.append(norm_ms)
        if trial < 2 or (trial+1) % 5 == 0:
            print(f"    trial {trial+1:3d}: total={total_ms:7.1f}ms, norm={norm_ms:6.2f}ms ({100*norm_ms/total_ms:5.2f}%)")
    prof.detach()
    sample_text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)[:150]
    free_model(model)
    # Stats
    total_sorted = sorted(total_ms_list); norm_sorted = sorted(norm_ms_list)
    median_total = total_sorted[len(total_sorted)//2]
    median_norm  = norm_sorted[len(norm_sorted)//2]
    fraction = median_norm / median_total
    summary = {
        "precision": precision,
        "median_total_ms": median_total,
        "median_norm_ms": median_norm,
        "fraction_norm": fraction,
        "tokens_per_sec": max_tokens / (median_total / 1000),
        "n_norm_modules": len(norms),
        "n_trials": n_trials,
        "max_tokens": max_tokens,
        "sample_output": sample_text,
        "all_total_ms": total_ms_list,
        "all_norm_ms": norm_ms_list,
    }
    print(f"  median: total={median_total:.1f}ms, norm={median_norm:.2f}ms ({fraction*100:.2f}%)")
    print(f"  tokens/sec: {summary['tokens_per_sec']:.2f}")
    return summary

SETUP_DONE = True
print("\nsetup complete. helpers loaded:")
print("  find_rmsnorm_modules, NormProfiler, load_at_precision, profile_precision, free_model")

# RMSNorm time-fraction profile across precisions

This notebook tests Nils's GGUF-angle claim empirically: as quantization gets more aggressive, the unquantized RMSNorm becomes a larger fraction of total decode time, so removing or fusing it has bigger relative impact.

**No architectural surgery, no Prop 3 patching.** Just profile a working RMSNorm model at three precisions and measure where time goes.

**Model:** `unsloth/Llama-3.2-1B-Instruct` (license-free unsloth mirror, standard RMSNorm pre-norm pattern, fits at all three precisions on A100).

**Precisions tested:**

1. bf16 (full precision baseline)
2. 8-bit BitsAndBytes
3. 4-bit BitsAndBytes (NF4, double quant)

**Output:** RMSNorm time as % of total decode time at each precision. If the % grows as precision drops, the GGUF angle is empirically supported.

**Wall-clock:** ~10 minutes total. Cost: ~5 Colab compute units.

## Run the experiment

In [ ]:
assert globals().get("SETUP_DONE", False), "Run cell 0 (the SETUP cell at the top) first."

# ---- Edit if you want ----
MODEL_ID    = "unsloth/Llama-3.2-1B-Instruct"
PRECISIONS  = ["bf16", "8bit", "4bit"]
N_WARMUP    = 3
N_TRIALS    = 20
MAX_TOKENS  = 256
PROMPT      = "Write a short story about a robot learning to paint."
# --------------------------

results = []
for precision in PRECISIONS:
    summary = profile_precision(MODEL_ID, precision, PROMPT, MAX_TOKENS, N_WARMUP, N_TRIALS)
    results.append(summary)

# Summary table
print("\n" + "="*72)
print(f"{'precision':>10} | {'tok/s':>9} | {'total ms':>10} | {'norm ms':>9} | {'norm %':>7}")
print("-"*72)
for r in results:
    print(f"{r['precision']:>10} | {r['tokens_per_sec']:>9.2f} | {r['median_total_ms']:>10.1f} | {r['median_norm_ms']:>9.2f} | {r['fraction_norm']*100:>6.2f}%")
print("="*72)

## Plot, save, download

In [ ]:
assert globals().get("SETUP_DONE", False), "Run cell 0 (the SETUP cell at the top) first."
assert 'results' in globals(), "Run the experiment cell first."

import matplotlib.pyplot as plt
import numpy as np

precisions    = [r['precision'] for r in results]
fractions_pct = [r['fraction_norm'] * 100 for r in results]
tps           = [r['tokens_per_sec'] for r in results]
norm_ms       = [r['median_norm_ms'] for r in results]
total_ms      = [r['median_total_ms'] for r in results]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: RMSNorm fraction (%)
ax = axes[0]
bars = ax.bar(precisions, fractions_pct, color=['tab:blue', 'tab:green', 'tab:orange'])
ax.set_ylabel('RMSNorm time / total decode time  (%)')
ax.set_title('RMSNorm time fraction across precisions')
for b, v in zip(bars, fractions_pct):
    ax.annotate(f'{v:.2f}%', xy=(b.get_x() + b.get_width()/2, v), ha='center', va='bottom', fontsize=10)
ax.grid(axis='y', alpha=0.3)

# Plot 2: total time and norm time
ax = axes[1]
x = np.arange(len(precisions))
width = 0.35
ax.bar(x - width/2, total_ms, width, label='total decode time', color='tab:blue', alpha=0.7)
ax.bar(x + width/2, norm_ms,  width, label='RMSNorm time',     color='tab:orange', alpha=0.7)
ax.set_xticks(x); ax.set_xticklabels(precisions)
ax.set_ylabel('time per generation (ms)')
ax.set_title(f'Decode time breakdown -- {MODEL_ID}')
ax.set_yscale('log')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.suptitle(f'FlashNorm GGUF-angle test: does RMSNorm fraction grow with quantization?\nGPU: {GPU_NAME}, max_tokens={MAX_TOKENS}, n_trials={N_TRIALS}', fontsize=11)
plt.tight_layout()
plt.savefig('rmsnorm_fraction_vs_precision.png', dpi=150, bbox_inches='tight')
plt.show()

# Verdict
print("\n=== verdict ===")
deltas = []
for i in range(1, len(fractions_pct)):
    delta = fractions_pct[i] - fractions_pct[i-1]
    deltas.append(delta)
    print(f"  {precisions[i-1]} -> {precisions[i]}: norm fraction {fractions_pct[i-1]:.2f}% -> {fractions_pct[i]:.2f}% (delta {delta:+.2f} pct points)")
if all(d > 0 for d in deltas):
    print(f"\nGGUF angle empirically supported: RMSNorm fraction grows monotonically with quantization aggressiveness.")
elif fractions_pct[-1] > fractions_pct[0]:
    print(f"\nGGUF angle partially supported: RMSNorm fraction is higher at {precisions[-1]} than at {precisions[0]} ({fractions_pct[-1]:.2f}% vs {fractions_pct[0]:.2f}%), but not strictly monotonic across all precisions.")
else:
    print(f"\nGGUF angle NOT supported on this hardware/model: RMSNorm fraction does not grow with quantization. Likely cause: BNB dequantize overhead inflates linear time at low precision on A100, masking the relative growth of norm cost.")

# Save
out = {
    "model": MODEL_ID,
    "gpu": GPU_NAME,
    "max_tokens": MAX_TOKENS,
    "n_trials": N_TRIALS,
    "n_warmup": N_WARMUP,
    "prompt": PROMPT,
    "precisions": PRECISIONS,
    "results": results,
}
RESULTS_PATH = Path("rmsnorm_fraction_results.json")
with open(RESULTS_PATH, 'w') as f:
    json.dump(out, f, indent=2, default=str)
print(f"\nresults saved to {RESULTS_PATH}")

# Auto-download
try:
    from google.colab import files
    files.download(str(RESULTS_PATH))
    files.download('rmsnorm_fraction_vs_precision.png')
except ImportError:
    print("not in Colab; files in working directory")

## Notes on what this measures

- **The number that matters is `fraction_norm`** (RMSNorm cumulative time / total decode time). It is what tests Nils's claim.
- The total decode time at lower precision may be **higher** than at bf16 on A100 due to BitsAndBytes dequantize overhead, but the **fraction** of that time spent in RMSNorm can still grow because the norm cost is mostly fixed while the linear cost varies.
- If the fraction does not grow on this hardware, it means BNB's overhead is masking the trend. The cleaner hardware for this measurement is Apple silicon with MLX, where 4-bit and 2-bit kernels are competitive with fp16. That is a follow-up if needed.
- A growing fraction is **direct empirical support** for the FlashNorm paper's GGUF-angle claim: as deployments quantize their linears more aggressively, the savings from removing or fusing the unquantized RMSNorm grow proportionally, even though absolute savings are bounded.